In [1]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
%matplotlib inline

이 셀은 **데이터 분석 및 머신러닝 모델 구축에 필요한 주요 라이브러리들을 임포트**합니다. `sklearn`을 통해 모델 선택, 선형 회귀, 평가 지표를 사용하고, `numpy`는 수치 계산, `matplotlib`는 시각화, `pandas`는 데이터 처리를 담당합니다. `%matplotlib inline`은 주피터 노트북에서 그림을 바로 표시하기 위한 설정입니다.

In [2]:
import os
# 노트북 파일이 있는 폴더로 이동 (예시)
os.chdir(r'C:\githome\hipython_rep')

# 변경 후 확인
print("변경 후:", os.getcwd())

변경 후: C:\githome\hipython_rep


데이터 파일의 경로를 설정하기 위해 **현재 작업 디렉토리를 변경하는 코드**입니다. `os.chdir()` 함수를 사용하여 지정된 경로로 이동하며, `os.getcwd()`를 통해 변경된 디렉토리를 확인합니다.

In [3]:
df = pd.read_csv('data1/Dubizzle_used_car_sales.csv')

중고차 가격 예측을 위한 **'Dubizzle_used_car_sales.csv' 데이터셋을 Pandas DataFrame으로 로드**합니다. 이 데이터는 중고차 가격 예측 모델 구축의 기반이 됩니다.

In [4]:
df.head()

,title,price_in_aed,kilometers,body_condition,mechanical_condition,seller_type,body_type,no_of_cylinders,transmission_type,regional_specs,horsepower,fuel_type,steering_side,year,color,emirate,motors_trim,company,model,date_posted
0,MITSUBISHI PAJERO 3.5L / 2013,26000,167390,Perfect inside and out,Perfect inside and out,Dealer,SUV,6,Automatic Transmission,GCC Specs,Unknown,Gasoline,Left Hand Side,2013.0,Silver,Dubai,GLS,mitsubishi,pajero,13/05/2022
1,chevrolet silverado,110000,39000,Perfect inside and out,Perfect inside and out,Dealer,SUV,8,Automatic Transmission,North American Specs,400 - 500 HP,Gasoline,Left Hand Side,2018.0,White,Sharjah,1500 High Country,chevrolet,silverado,14/01/2022
2,MERCEDES-BENZ E300 - 2014 - GCC SPEC - FULL OP...,78000,200000,Perfect inside and out,Perfect inside and out,Dealer,Sedan,6,Automatic Transmission,GCC Specs,400 - 500 HP,Gasoline,Left Hand Side,2014.0,Blue,Sharjah,E 300,mercedes-benz,e-class,05/05/2022
3,WARRANTY UNTIL APR 2023 || Ferrari 488 Spider ...,899000,27000,Perfect inside and out,Perfect inside and out,Dealer,Hard Top Convertible,8,Automatic Transmission,GCC Specs,600 - 700 HP,Gasoline,Left Hand Side,2018.0,Red,Dubai,Standard,ferrari,488-spider,30/04/2022
4,USED RENAULT DOKKER 2020,33000,69000,Perfect inside and out,Perfect inside and out,Owner,Wagon,4,Manual Transmission,GCC Specs,Less than 150 HP,Gasoline,Left Hand Side,2020.0,White,Dubai,Standard,renault,dokker,13/05/2022


로드된 데이터프레임 `df`의 **상위 5행을 출력하여 데이터의 초기 구조와 내용을 빠르게 확인**합니다. 각 컬럼의 데이터 형식과 포함된 값들을 엿볼 수 있습니다.

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9970 entries, 0 to 9969
Data columns (total 20 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   title                 9965 non-null   object 
 1   price_in_aed          9970 non-null   int64  
 2   kilometers            9970 non-null   int64  
 3   body_condition        9970 non-null   object 
 4   mechanical_condition  9970 non-null   object 
 5   seller_type           9970 non-null   object 
 6   body_type             9970 non-null   object 
 7   no_of_cylinders       9889 non-null   object 
 8   transmission_type     9970 non-null   object 
 9   regional_specs        9970 non-null   object 
 10  horsepower            9970 non-null   object 
 11  fuel_type             9970 non-null   object 
 12  steering_side         9970 non-null   object 
 13  year                  9000 non-null   float64
 14  color                 9970 non-null   object 
 15  emirate              

로드된 데이터프레임 `df`의 **기본 정보를 확인**합니다. 각 컬럼의 이름, Non-Null 값의 개수, 데이터 타입(Dtype), 그리고 메모리 사용량을 통해 데이터의 구조와 누락된 값 여부를 빠르게 파악할 수 있습니다.

In [6]:
# 결측치 개수
missing_count = df.isnull().sum()

# 전체 대비 결측치 비율(%)
missing_ratio = (missing_count / len(df)) * 100

# 결측치 요약표 출력
missing_df = pd.DataFrame({
    'Missing Count': missing_count,
    'Missing Ratio (%)': missing_ratio
}).sort_values(by='Missing Count', ascending=False)

missing_df[missing_df['Missing Count'] > 0]

,Missing Count,Missing Ratio (%)
year,970,9.729188
no_of_cylinders,81,0.812437
motors_trim,28,0.280843
title,5,0.050150


데이터프레임 내 **결측치의 개수와 전체 데이터 대비 비율을 계산하여 요약표로 출력**합니다. `.isnull().sum()`을 통해 결측치 개수를 파악하고, 이를 정렬하여 어떤 컬럼에 결측치가 많은지 직관적으로 확인합니다.

In [7]:
no_null_df = df.copy()
no_null_df['no_of_cylinders'].fillna(no_null_df['no_of_cylinders'].mode()[0],inplace=True)
no_null_df['motors_trim'].fillna('unknown',inplace=True)
no_null_df['title'].fillna('unknown',inplace=True)
no_null_df['year'].fillna(no_null_df['year'].median(),inplace=True)

C:\Users\Admin\AppData\Local\Temp\ipykernel_16196\3555952243.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  no_null_df['no_of_cylinders'].fillna(no_null_df['no_of_cylinders'].mode()[0],inplace=True)
C:\Users\Admin\AppData\Local\Temp\ipykernel_16196\3555952243.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting

결측치를 처리하기 위해 **`fillna`를 사용하여 각 컬럼의 누락된 값을 채웁니다.** 'no_of_cylinders'는 최빈값으로, 'year'는 중앙값으로 대체하며, 'motors_trim'과 'title'은 'unknown'이라는 문자열로 채워 데이터의 완전성을 높입니다.

In [8]:
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
le = LabelEncoder()

범주형 데이터 처리를 위한 **`LabelEncoder`와 수치형 데이터의 스케일 조정을 위한 `StandardScaler`를 `sklearn.preprocessing` 모듈에서 임포트**합니다. 이는 머신러닝 모델의 성능 향상을 위한 필수적인 전처리 단계입니다.

In [9]:
encoded_df = no_null_df.copy()
for col in encoded_df.columns:
    if encoded_df[col].dtype == 'object':
        encoded_df[col] = le.fit_transform(encoded_df[col])

데이터프레임의 **모든 `object` 타입(문자열) 컬럼을 `LabelEncoder`를 사용하여 수치형으로 변환**합니다. 이 과정을 통해 컴퓨터가 이해할 수 있는 형태로 데이터를 가공하여 모델 학습에 사용될 수 있도록 준비합니다.

In [10]:
encoded_df.head()

,title,price_in_aed,kilometers,body_condition,mechanical_condition,seller_type,body_type,no_of_cylinders,transmission_type,regional_specs,horsepower,fuel_type,steering_side,year,color,emirate,motors_trim,company,model,date_posted
0,6507,26000,167390,3,4,0,6,5,0,1,10,2,0,2013.0,12,3,386,51,371,154
1,8929,110000,39000,3,4,0,6,6,0,3,3,2,0,2018.0,15,6,19,10,457,160
2,6451,78000,200000,3,4,0,7,5,0,1,3,2,0,2014.0,2,6,296,46,177,54
3,8823,899000,27000,3,4,0,2,6,0,1,5,2,0,2018.0,11,3,723,16,30,333
4,8614,33000,69000,3,4,2,12,3,1,1,9,2,0,2020.0,15,3,723,59,172,154


**레이블 인코딩이 완료된 데이터프레임 `encoded_df`의 상위 5행을 출력**하여, 문자열 데이터가 숫자로 정상적으로 변환되었는지 확인합니다.

In [11]:
X = encoded_df.drop('price_in_aed',axis=1).values
y = encoded_df['price_in_aed']

예측하고자 하는 **'price_in_aed' 컬럼을 타겟 변수 `y`로 설정하고, 이를 제외한 나머지 모든 컬럼들을 피처(독립 변수) `X`로 분리**합니다. 이는 머신러닝 모델 학습을 위한 표준적인 데이터 준비 과정입니다.

In [12]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=50)
scaler = StandardScaler()
scaler.fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

전처리된 데이터를 **훈련 세트와 테스트 세트로 8:2 비율로 분리**합니다. `random_state=50`은 재현 가능한 결과를 보장합니다. 이후 `StandardScaler`를 사용하여 훈련 세트(`X_train`)에 맞춰 스케일링을 학습하고, 이 스케일링을 훈련 및 테스트 세트(`X_train_scaled`, `X_test_scaled`)에 적용합니다.

In [13]:
lr = LinearRegression()
lr.fit(X_train_scaled,y_train)
y_pred = lr.predict(X_test_scaled)

이 셀은 **선형 회귀 모델(`LinearRegression`)을 초기화하고, 스케일링된 훈련 데이터(`X_train_scaled`, `y_train`)를 사용하여 모델을 학습**시킵니다. 학습된 모델은 `fit()` 메서드를 통해 데이터의 패턴을 파악하고, `predict()` 메서드를 사용하여 스케일링된 테스트 데이터(`X_test_scaled`)에 대한 중고차 가격 예측값(`y_pred`)을 생성합니다.

In [14]:
mse = mean_squared_error(y_test,y_pred)
rmse = np.sqrt(mse)

rmse, mse

(np.float64(409321.05529370083), np.float64(167543726306.74887))

이 셀은 선형 회귀 모델의 **예측 성능을 평가하기 위해 평균 제곱 오차(MSE)와 제곱근 평균 제곱 오차(RMSE)를 계산**합니다. RMSE는 예측 오류의 크기를 실제 값과 동일한 단위로 나타내므로 모델의 예측 정확도를 직관적으로 이해하는 데 유용합니다.

In [15]:
r2_score(y_test, y_pred)

np.float64(0.15872922130211664)

이 셀은 선형 회귀 모델의 **결정 계수(R2 스코어)를 계산하여 모델의 설명력을 평가**합니다. `r2_score` 함수는 모델이 종속 변수(`y_test`)의 분산을 얼마나 잘 설명하는지를 나타냅니다.

In [16]:
pd.Series(data = np.abs(np.round(lr.coef_,1)), index=encoded_df.drop('price_in_aed',axis=1).columns).sort_values(ascending=False)

horsepower              82182.7
year                    78162.9
seller_type             64038.4
color                   40602.1
regional_specs          33912.7
company                 25696.3
motors_trim             19976.4
transmission_type       19371.3
fuel_type               17476.9
no_of_cylinders         17454.3
title                   16354.9
model                   15079.8
body_type               14385.2
emirate                 13598.3
body_condition           7689.4
kilometers               5585.8
mechanical_condition     4310.7
steering_side            2413.6
date_posted              1132.5
dtype: float64

이 셀은 **선형 회귀 모델의 회귀 계수(`lr.coef_`)를 활용하여 각 피처의 중요도를 분석**합니다. 회귀 계수의 절댓값을 사용하여 피처가 타겟 변수에 미치는 영향의 크기를 파악하고, 이를 Pandas Series로 변환한 후 내림차순으로 정렬하여 어떤 피처가 가격 예측에 큰 영향을 미치는지 확인합니다.

In [17]:
X = encoded_df.drop(['price_in_aed', 'date_posted', 'steering_side'],axis=1).values
y = encoded_df['price_in_aed']

이 셀은 **앞서 분석한 피처 중요도 결과를 바탕으로 영향력이 적은 'date_posted', 'steering_side' 컬럼을 추가로 제외하여 피처 `X`를 재설정**합니다. 이는 모델을 단순화하고 성능 개선을 시도하기 위함입니다.

In [18]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=50)
scaler = StandardScaler()
scaler.fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

이 셀은 **재설정된 피처 `X`와 타겟 `y`를 사용하여 데이터를 다시 훈련 세트와 테스트 세트로 분리하고 스케일링**합니다. `train_test_split`을 통해 8:2 비율로 분리하며 `random_state=50`을 유지하여 일관성을 확보합니다. 이 과정은 변경된 피처 셋으로 모델을 재학습하기 위한 필수적인 데이터 전처리입니다.

In [19]:
lr = LinearRegression()
lr.fit(X_train_scaled,y_train)
y_pred = lr.predict(X_test_scaled)

이 셀은 **재조정된 데이터(`X_train_scaled`, `y_train`)를 사용하여 선형 회귀 모델을 다시 학습**하고 예측을 수행합니다. 피처 선택이 모델 성능에 미치는 영향을 평가할 수 있습니다.

In [20]:
mse = mean_squared_error(y_test,y_pred)
rmse = np.sqrt(mse)

rmse, mse

(np.float64(409330.9773900221), np.float64(167551849051.07077))

이 셀은 피처가 조정된 후 **재학습된 선형 회귀 모델의 예측 성능을 평가하기 위해 MSE와 RMSE를 다시 계산**합니다. 이전 모델의 평가 결과와 비교하여, 컬럼 제거가 모델의 오차 크기에 어떤 영향을 미쳤는지 확인합니다.

In [21]:
r2_score(y_test, y_pred)

np.float64(0.15868843536765354)

이 셀은 피처가 조정된 후 **재학습된 선형 회귀 모델의 결정 계수(R2 스코어)를 계산하여 모델의 설명력을 재평가**합니다. 이전 모델의 R2 스코어와 비교하여, 피처 선택의 변화가 모델의 설명력에 미친 영향을 분석합니다.

In [22]:
from sklearn.model_selection import cross_val_score
neg_mse_scores = cross_val_score(lr, X, y, scoring='neg_mean_squared_error', cv=5)
neg_mse_scores

array([-2.01521777e+13, -1.45012509e+11, -2.48066673e+11, -1.51168493e+11,
       -1.43765720e+11])

이 셀은 **교차 검증(`cross_val_score`)을 사용하여 선형 회귀 모델의 평균 제곱 오차(MSE)를 평가**합니다. `scoring='neg_mean_squared_error'`를 사용하여 음의 MSE 값을 얻고, 이를 통해 모델의 일반화 성능을 여러 폴드에서 확인합니다.

In [23]:
RMSE = np.sqrt(neg_mse_scores * (-1))
np.mean(RMSE), RMSE

(np.float64(1227190.8262765508),
 array([4489117.70182181,  380805.07959723,  498062.921147  ,
         388803.92671001,  379164.50210671]))

이 셀은 **교차 검증으로 얻은 음의 MSE 값으로부터 RMSE를 계산**하고, 그 평균과 각 폴드별 RMSE 값을 확인합니다. 이를 통해 모델의 예측 오차 수준을 보다 신뢰성 있게 파악할 수 있습니다.

In [24]:
r2_scores = cross_val_score(lr, X, y, scoring='r2', cv=5)
r2_scores, np.mean(r2_scores)

(array([-8.36270042e+01,  1.56973068e-01,  7.97636296e-02,  1.49351633e-01,
         1.60964525e-01]),
 np.float64(-16.615990261965102))

이 셀은 **교차 검증(`cross_val_score`)을 사용하여 선형 회귀 모델의 결정 계수(R2 스코어)를 평가**합니다. 각 폴드별 R2 스코어와 그 평균을 확인하여 모델의 설명력과 일반화 성능을 종합적으로 판단합니다.

In [25]:
from sklearn.ensemble import RandomForestRegressor
rf = RandomForestRegressor(random_state=42, max_depth=8)
rf.fit(X_train_scaled,y_train)
y_pred = rf.predict(X_test_scaled)

이 셀은 **앙상블 모델인 `RandomForestRegressor`를 초기화하고 학습**시킵니다. `random_state=42`는 결과의 재현성을 위해 설정되었으며, `max_depth=8`은 트리의 최대 깊이를 제한하여 과적합을 방지합니다. 학습된 모델은 스케일링된 테스트 데이터(`X_test_scaled`)에 대한 중고차 가격을 예측합니다.

In [26]:
mse = mean_squared_error(y_test,y_pred)
rmse = np.sqrt(mse)

rmse, mse

(np.float64(214068.31901716476), np.float64(45825245206.834625))

이 셀은 **랜덤 포레스트 회귀 모델의 예측 성능을 평가하기 위해 MSE와 RMSE를 계산**합니다. 선형 회귀 모델의 결과와 비교하여, 랜덤 포레스트가 가격 예측에 있어 어떤 성능을 보이는지 확인합니다.

In [27]:
r2_score(y_test, y_pred)

np.float64(0.7699022185492461)

이 셀은 **랜덤 포레스트 회귀 모델의 결정 계수(R2 스코어)를 계산하여 모델의 설명력을 평가**합니다. 선형 회귀 모델의 R2 스코어와 비교하여, 앙상블 모델의 예측력이 얼마나 향상되었는지 파악합니다.

In [28]:
neg_mse_scores = cross_val_score(rf, X, y, scoring='neg_mean_squared_error', cv=5)
neg_mse_scores

array([-5.37935057e+10, -3.83944620e+10, -1.08023877e+11, -4.44078856e+10,
       -2.00488039e+10])

이 셀은 **교차 검증을 통해 랜덤 포레스트 회귀 모델의 평균 제곱 오차(MSE)를 평가**합니다. 여러 폴드에서 얻은 음의 MSE 값을 통해 모델의 안정적인 일반화 성능을 확인합니다.

In [29]:
RMSE = np.sqrt(neg_mse_scores * (-1))
np.mean(RMSE), RMSE

(np.float64(221774.95280178468),
 array([231934.27023239, 195945.04831144, 328669.86067238, 210731.78586989,
        141593.79892281]))

이 셀은 **교차 검증된 음의 MSE 값으로부터 RMSE를 계산**하고, 그 평균과 각 폴드별 RMSE 값을 확인합니다. 이는 랜덤 포레스트 모델의 예측 오차 수준에 대한 보다 신뢰성 있는 지표를 제공합니다.

In [30]:
r2_scores = cross_val_score(rf, X, y, scoring='r2', cv=5)
r2_scores, np.mean(r2_scores)

(array([0.77409969, 0.77679467, 0.59927104, 0.75011   , 0.88299257]),
 np.float64(0.7566535910922498))

이 셀은 **교차 검증을 사용하여 랜덤 포레스트 회귀 모델의 결정 계수(R2 스코어)를 평가**합니다. 각 폴드별 R2 스코어와 그 평균을 확인하여 랜덤 포레스트 모델의 전반적인 설명력과 일반화 성능을 파악합니다.

In [31]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures

`sklearn.pipeline`의 `Pipeline`은 여러 처리 단계를 하나로 묶어 워크플로우를 효율화하며, `sklearn.preprocessing`의 `PolynomialFeatures`는 데이터에 다항 특성을 추가하여 모델의 비선형 학습 능력을 높입니다.

In [32]:
results = []
for degree in range(1,6):
    model_poly = Pipeline([
        ('poly',PolynomialFeatures(degree=degree, include_bias=False)),
        ('linear',LinearRegression())
    ])
    model_poly.fit(X_train_scaled, y_train)
    pred_poly = model_poly.predict(X_test_scaled)
    mse = mean_squared_error(y_test,pred_poly)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, pred_poly)
    results.append({
        'degree' : degree,
        'MSE' : mse,
        'RMSE' : rmse,
        'R2' : r2
    })
pd.DataFrame(results)

,degree,MSE,RMSE,R2
0,1,1.675518e+11,4.093310e+05,1.586884e-01
1,2,9.259381e+10,3.042923e+05,5.350678e-01
2,3,1.809893e+12,1.345323e+06,-8.087837e+00
3,4,1.211254e+15,3.480308e+07,-6.080951e+03
4,5,4.921698e+19,7.015482e+09,-2.471284e+08


이 셀은 `Pipeline`을 사용하여 **다항 특성 변환(`PolynomialFeatures`)과 선형 회귀 모델(`LinearRegression`)을 결합**하고, **다양한 다항식 차수(1차부터 5차까지)에 따라 모델을 학습하고 성능을 평가**합니다. 각 차수별로 MSE, RMSE, R2 스코어를 계산하여 DataFrame으로 출력하고 **다항 회귀 모델의 차수에 따른 성능 변화를 분석**합니다.

In [33]:
results = []
best_r2 = False
for degree in range(2,3):
    model_poly = Pipeline([
        ('poly',PolynomialFeatures(degree=degree, include_bias=False)),
        ('RF',RandomForestRegressor(random_state=22, max_depth=8))
    ])
    model_poly.fit(X_train_scaled, y_train)
    pred_poly = model_poly.predict(X_test_scaled)
    mse = mean_squared_error(y_test,pred_poly)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, pred_poly)
    results.append({
        'degree' : degree,
        'MSE' : mse,
        'RMSE' : rmse,
        'R2' : r2
    })
    if best_r2 < r2 or best_r2 == False:
        best_r2 = r2
        best_model = model_poly
        best_pred_poly = pred_poly
pd.DataFrame(results)

,degree,MSE,RMSE,R2
0,2,3.046984e+10,174556.119275,0.847005


이 셀은 `Pipeline`을 사용하여 **2차 다항 특성 변환(`PolynomialFeatures`)과 랜덤 포레스트 회귀 모델(`RandomForestRegressor`)을 결합**하여 모델을 학습하고 성능을 평가합니다. 이 코드는 **가장 높은 R2 스코어를 기록한 모델(`best_model`)과 해당 예측값(`best_pred_poly`)을 저장**하여 최적의 모델을 찾는 과정을 포함합니다. 최종적으로 결과를 DataFrame으로 출력하여 성능을 확인합니다.<br>
(이 부분의 경우 1부터 5차를 검사했을때 2차가 가장 높았으며 결과를 빠르게 내기 위해 2차만 검사했습니다)

In [34]:
from sklearn.ensemble import GradientBoostingRegressor

부스팅 계열의 앙상블 모델인 `GradientBoostingRegressor`를 임포트합니다.

In [35]:
gb_clf = GradientBoostingRegressor()
gb_clf.fit(X_train_scaled,y_train)
gb_pred = gb_clf.predict(X_test_scaled)

이 셀은 **그레디언트 부스팅 회귀 모델(`GradientBoostingRegressor`)을 초기화하고 학습**시킵니다. 학습된 모델은 스케일링된 훈련 데이터로 피팅되며, 이후 테스트 데이터에 대한 예측을 수행합니다.

In [36]:
mse = mean_squared_error(y_test,gb_pred)
rmse = np.sqrt(mse)

rmse, mse

(np.float64(202983.94520426422), np.float64(41202482010.68774))

이 셀은 **그레디언트 부스팅 회귀 모델의 예측 성능을 평가하기 위해 MSE와 RMSE를 계산**합니다. 이는 모델의 예측 정확도를 나타내는 중요한 지표입니다.

In [37]:
r2_score(y_test, gb_pred)

np.float64(0.7931140431844355)

이 셀은 **그레디언트 부스팅 회귀 모델의 결정 계수(R2 스코어)를 계산**하여 모델의 설명력을 평가합니다.

In [38]:
from xgboost import XGBRegressor

고성능 앙상블 모델인 `XGBRegressor`를 임포트합니다.

In [39]:
xgb = XGBRegressor(n_estimators=400, learning_rate=0.1, max_depth=3, use_label_encoder=False)
evals = [(X_test_scaled, y_test)]
xgb.fit(X_train_scaled, y_train, early_stopping_rounds=40, 
        eval_set=evals, verbose=True)
xgb_pred = xgb.predict(X_test_scaled)

[0]	validation_0-rmse:478167.70972
[1]	validation_0-rmse:447694.51630
[2]	validation_0-rmse:419264.46199
[3]	validation_0-rmse:395406.77145
[4]	validation_0-rmse:371141.70240
[5]	validation_0-rmse:351953.98151
[6]	validation_0-rmse:335276.80090
[7]	validation_0-rmse:319646.59413
[8]	validation_0-rmse:305922.31344
[9]	validation_0-rmse:295346.27887
[10]	validation_0-rmse:285859.57459
[11]	validation_0-rmse:276269.90212
[12]	validation_0-rmse:267949.71339
[13]	validation_0-rmse:260519.38516
[14]	validation_0-rmse:256210.49826
[15]	validation_0-rmse:253834.11236
[16]	validation_0-rmse:249437.02606
[17]	validation_0-rmse:249290.28136
[18]	validation_0-rmse:248206.89043
[19]	validation_0-rmse:246499.51528
[20]	validation_0-rmse:243825.34987
[21]	validation_0-rmse:245522.15169
[22]	validation_0-rmse:246840.31339
[23]	validation_0-rmse:245121.61758
[24]	validation_0-rmse:246121.39911
[25]	validation_0-rmse:249633.40996
[26]	validation_0-rmse:251482.80684
[27]	validation_0-rmse:250835.67459
[2

c:\Users\Admin\miniconda3\envs\hi_ml_env\lib\site-packages\xgboost\sklearn.py:793: UserWarning: `early_stopping_rounds` in `fit` method is deprecated for better compatibility with scikit-learn, use `early_stopping_rounds` in constructor or`set_params` instead.
  warnings.warn(


[44]	validation_0-rmse:239375.48827
[45]	validation_0-rmse:237841.86260
[46]	validation_0-rmse:237368.13050
[47]	validation_0-rmse:237282.20363
[48]	validation_0-rmse:235993.37847
[49]	validation_0-rmse:235357.11577
[50]	validation_0-rmse:235040.72850
[51]	validation_0-rmse:235026.17884
[52]	validation_0-rmse:234039.08216
[53]	validation_0-rmse:233204.73821
[54]	validation_0-rmse:232677.70656
[55]	validation_0-rmse:232538.22536
[56]	validation_0-rmse:232403.80428
[57]	validation_0-rmse:232658.06991
[58]	validation_0-rmse:232293.96285
[59]	validation_0-rmse:231921.13990
[60]	validation_0-rmse:231806.38320
[61]	validation_0-rmse:231195.18976
[62]	validation_0-rmse:230735.49464
[63]	validation_0-rmse:230679.12657
[64]	validation_0-rmse:230093.61867
[65]	validation_0-rmse:229005.37120
[66]	validation_0-rmse:228879.04791
[67]	validation_0-rmse:229105.16635
[68]	validation_0-rmse:228926.23374
[69]	validation_0-rmse:228347.49662
[70]	validation_0-rmse:228061.47531
[71]	validation_0-rmse:22865

이 셀은 **XGBoost(`XGBRegressor`) 모델을 초기화하고, 조기 종료(early stopping) 옵션을 사용하여 학습**시킵니다. `n_estimators`, `learning_rate`, `max_depth` 등의 하이퍼파라미터를 설정하고, 검증 세트(`eval_set`)를 이용해 과적합을 방지하며 최적의 성능으로 모델을 학습시킵니다.

In [40]:
mse = mean_squared_error(y_test,xgb_pred)
rmse = np.sqrt(mse)

rmse, mse

(np.float64(211608.02865231445), np.float64(44777957790.11873))

이 셀은 **XGBoost 모델의 예측 성능을 평가하기 위해 MSE와 RMSE를 계산**합니다.

In [41]:
r2_score(y_test, xgb_pred)

np.float64(0.7751608595022831)

이 셀은 **XGBoost 모델의 결정 계수(R2 스코어)를 계산**하여 모델의 설명력을 평가합니다.

In [42]:
from sklearn.linear_model import Ridge, Lasso, ElasticNet

L2 및 L1 규제를 사용하는 선형 회귀 모델인 **`Ridge`, `Lasso`, `ElasticNet`을 임포트**합니다.

In [43]:
ridge = Ridge(alpha=10)
ridge.fit(X_train_scaled,y_train)
pred_ridge = ridge.predict(X_test_scaled)

mse = mean_squared_error(y_test, pred_ridge)
r2 = r2_score(y_test, pred_ridge)
mse, r2

(np.float64(167560241054.94778), np.float64(0.1586462974267585))

릿지(Ridge) 회귀 모델을 **알파(alpha) 값 10**으로 설정하여 학습하고, 테스트 데이터에 대한 예측을 수행합니다. 이어서 예측 결과와 실제 값(`y_test`)을 비교하여 **평균 제곱 오차(MSE)**와 **결정 계수($R^2$)**를 계산하고 출력합니다.

In [44]:
from sklearn.linear_model import RidgeCV, LassoCV
alphas = [0.001, 0.01, 0.1, 1, 10, 100]
ridge_cv = RidgeCV(alphas=alphas, cv=5)
ridge_cv.fit(X_train_scaled, y_train)
ridge_preds = ridge_cv.predict(X_test_scaled)
ridge_mse = mean_squared_error(y_test, ridge_preds)
ridge_r2 = r2_score(y_test, ridge_preds)
print(f'ridge cv mse : {ridge_mse:.4f}, r2 : {ridge_r2:.4f}')

ridge cv mse : 167637697862.9581, r2 : 0.1583


`RidgeCV`를 사용하여 교차 검증을 통해 최적의 알파(alpha) 값을 찾는 릿지(Ridge) 회귀 모델을 학습시킵니다. 이후 테스트 데이터에 대한 예측을 수행하고, 모델의 성능을 **평균 제곱 오차(MSE)**와 **결정 계수($R^2$)**로 평가하여 출력합니다.

In [45]:
ridge_cv.alpha_

np.float64(100.0)

`ridge_cv.alpha_`는 `RidgeCV` 모델이 교차 검증을 통해 **자동으로 찾아낸 최적의 알파(alpha) 값**을 나타냅니다. 이 값은 모델 학습 시 가장 좋은 성능을 보인 정규화 강도를 의미합니다.

In [46]:
lasso = Lasso(alpha=0.1)
lasso.fit(X_train_scaled,y_train)
pred_lasso = lasso.predict(X_test_scaled)

mse = mean_squared_error(y_test, pred_lasso)
r2 = r2_score(y_test, pred_lasso)
mse, r2

(np.float64(167551881774.66205), np.float64(0.158688271055914))

라쏘(Lasso) 회귀 모델을 **알파(alpha) 값 0.1**로 설정하여 학습하고, 테스트 데이터에 대한 예측을 수행합니다. 이어서 예측 결과와 실제 값(`y_test`)을 비교하여 **평균 제곱 오차(MSE)**와 **결정 계수($R^2$)**를 계산하고 출력합니다.

In [47]:
alphas = [0.001, 0.01, 0.1, 1, 10, 100]
lasso_cv = LassoCV(alphas=alphas, cv=5)
lasso_cv.fit(X_train_scaled, y_train)
lasso_preds = lasso_cv.predict(X_test_scaled)
lasso_mse = mean_squared_error(y_test, lasso_preds)
lasso_r2 = r2_score(y_test, lasso_preds)
print(f'ridge cv mse : {lasso_mse:.4f}, r2 : {lasso_r2:.4f}')

ridge cv mse : 167584767470.7297, r2 : 0.1585


`LassoCV`를 사용하여 교차 검증을 통해 최적의 알파(alpha) 값을 찾는 라쏘(Lasso) 회귀 모델을 학습시킵니다. 이후 테스트 데이터에 대한 예측을 수행하고, 모델의 성능을 **평균 제곱 오차(MSE)**와 **결정 계수($R^2$)**로 평가하여 출력합니다.

In [48]:
lasso_cv.alpha_

np.float64(100.0)

`lasso_cv.alpha_`는 `LassoCV` 모델이 교차 검증을 통해 **자동으로 찾아낸 최적의 알파(alpha) 값**을 나타냅니다.

In [49]:
lasso_cv.coef_

array([-16277.57681122,  -5490.27381924,   7543.13144382,  -4217.15727687,
       -63832.84915491, -14293.03686434, -17404.77563772, -19172.10527723,
       -33854.80638735,  82192.29768454,  17403.98852137,  77893.94706602,
       -40443.33837397, -13547.68598914,  19796.54432914,  25510.59910283,
       -14959.0713687 ])

`lasso_cv.coef_`는 `LassoCV` 모델이 학습 후 **추정한 특성(feature)들의 회귀 계수(coefficients)**를 나타냅니다. 라쏘(Lasso) 회귀의 특성상 이 값들 중 일부는 **0이 되어 특정 특성들이 모델에서 제외**될 수 있습니다.

In [50]:
ridge_cv.coef_

array([-16251.87916552,  -5572.88367823,   7760.45619837,  -4172.44172747,
       -63247.34210821, -14216.84175454, -17318.85780819, -19074.23114754,
       -33534.66068186,  81266.69000361,  17255.27792898,  77100.79146435,
       -40039.79719434, -13477.75028629,  19681.09497479,  25267.44561449,
       -14850.98982586])

`ridge_cv.coef_`는 `RidgeCV` 모델이 학습 후 **추정한 각 특성(feature)들의 회귀 계수(coefficients)**를 나타내며, 라쏘 모델의 계수와 비교하여 규제 방식의 차이를 확인할 수 있습니다.

In [51]:
enet = ElasticNet(alpha=0.1, l1_ratio=0.5)
enet.fit(X_train_scaled,y_train)

ElasticNet(alpha=0.1)

엘라스틱넷(ElasticNet) 회귀 모델을 **알파(alpha) 값 0.1**과 **L1_ratio 값 0.5**로 설정하여 학습시킵니다. 엘라스틱넷은 L1 규제와 L2 규제를 모두 사용하는 모델입니다.

In [52]:
enet_pred = enet.predict(X_test_scaled)

학습된 엘라스틱넷(ElasticNet) 모델 `enet`을 사용하여 **스케일링된 테스트 데이터(`X_test_scaled`)에 대한 예측값 `enet_pred`를 생성**합니다.

In [53]:
mse = mean_squared_error(y_test, enet_pred)
r2 = r2_score(y_test, enet_pred)
mse, r2

(np.float64(167917003545.75412), np.float64(0.15685492113910604))

엘라스틱넷(ElasticNet) 모델의 예측(`enet_pred`)과 실제 값(`y_test`)을 비교하여 **평균 제곱 오차(MSE)**와 **결정 계수($R^2$)**를 계산하고 출력합니다.

In [54]:
results = pd.DataFrame({
    '모델' : ['다항회귀', 'XGB', '릿지회귀', '라쏘회귀', '엘라스틱넷회귀'],
    'RMSE' : [np.sqrt(mean_squared_error(y_test,best_pred_poly)),
             np.sqrt(mean_squared_error(y_test,xgb_pred)),
             np.sqrt(mean_squared_error(y_test,pred_ridge)),
             np.sqrt(mean_squared_error(y_test,pred_lasso)),
             np.sqrt(mean_squared_error(y_test,enet_pred))],
    'R2' : [r2_score(y_test,best_pred_poly),
            r2_score(y_test,xgb_pred),
            r2_score(y_test,pred_ridge),
            r2_score(y_test,pred_lasso),
            r2_score(y_test,enet_pred)]
})
results

,모델,RMSE,R2
0,다항회귀,174556.119275,0.847005
1,XGB,211608.028652,0.775161
2,릿지회귀,409341.228140,0.158646
3,라쏘회귀,409331.017362,0.158688
4,엘라스틱넷회귀,409776.772824,0.156855


여러 회귀 모델(다항회귀, XGB, 릿지회귀, 라쏘회귀, 엘라스틱넷회귀)의 성능 지표인 **RMSE(평균 제곱근 오차)**와 **R2(결정 계수)**를 취합하여 **`results`라는 데이터프레임을 생성하고 출력**합니다. 이를 통해 각 모델의 성능을 한눈에 비교하고 최적의 모델을 선택할 수 있습니다.